# N01 — Classical NLP: Text Preprocessing & Feature Engineering

## Why classical NLP still matters

LLMs dominate headlines, but classical NLP methods are still:
- The baseline you must beat to justify a complex model
- The only option when compute is constrained
- Tested in data science interviews more than transformer architectures
- The foundation for understanding *why* embeddings work

## The NLP pipeline

```
Raw Text → Tokenization → Normalization → Feature Extraction → Model
```

Each step has decisions that change your model's performance significantly.

**Reference:** [spaCy docs](https://spacy.io/api) | [sklearn text docs](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction)

**Allowed:** `spacy`, `sklearn`, `numpy`, `pandas`


In [ ]:
import spacy
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

# 20 Newsgroups: 4 categories for binary/multiclass exercises
categories = ['sci.med', 'sci.space', 'rec.sport.hockey', 'talk.politics.guns']
news_train = fetch_20newsgroups(subset='train', categories=categories,
                                  remove=('headers','footers','quotes'))
news_test  = fetch_20newsgroups(subset='test',  categories=categories,
                                  remove=('headers','footers','quotes'))

X_train_raw = news_train.data
y_train     = news_train.target
X_test_raw  = news_test.data
y_test      = news_test.target
target_names = news_train.target_names

print(f"Train: {len(X_train_raw)} docs | Test: {len(X_test_raw)} docs")
print(f"Classes: {target_names}")
print(f"\nSample doc (first 200 chars):")
print(X_train_raw[0][:200])

---
## Exercise 1 — Text Preprocessing Pipeline

**Task:** Build a reusable text cleaning function and understand what each step does.

Implement `preprocess_text(text, config)` where `config` is a dict of boolean flags:
- `lowercase`: convert to lowercase
- `remove_numbers`: strip numeric tokens
- `remove_punctuation`: remove non-alphanumeric chars
- `remove_stopwords`: remove spaCy stopwords
- `lemmatize`: replace tokens with their lemma (spaCy)
- `min_token_len`: minimum token length to keep (int)

Apply to a batch of documents. Return cleaned strings.

**Then:** Show how each preprocessing step changes the vocabulary size — this teaches you what each step actually does.

In [ ]:
DEFAULT_CONFIG = {
    'lowercase': True,
    'remove_numbers': True,
    'remove_punctuation': True,
    'remove_stopwords': True,
    'lemmatize': True,
    'min_token_len': 3
}

def preprocess_text(text: str, config: dict = None) -> str:
    """
    Configurable text preprocessing pipeline using spaCy.
    Returns cleaned string.
    """
    if config is None:
        config = DEFAULT_CONFIG
    # YOUR CODE HERE
    # 1. Parse with spaCy: doc = nlp(text)
    # 2. For each token in doc, apply filters
    # 3. Return ' '.join(filtered_tokens)
    pass

def preprocess_batch(texts: list, config: dict = None, batch_size: int = 100) -> list:
    """
    Process a list of texts using nlp.pipe for efficiency.
    """
    # YOUR CODE HERE
    # Use nlp.pipe(texts, batch_size=batch_size) for speed
    pass

# Test on first 500 training docs
sample = X_train_raw[:500]
cleaned = preprocess_batch(sample, DEFAULT_CONFIG)
print(f"Before: {sample[0][:100]}")
print(f"After:  {cleaned[0][:100]}")

In [ ]:
# --- ASSERTIONS ---
assert cleaned is not None and len(cleaned) == 500
assert all(isinstance(t, str) for t in cleaned)

# Lowercase check
assert all(t == t.lower() for t in cleaned if t), "Must be lowercase"

# Stopwords removed: common words should not appear
stopwords = nlp.Defaults.stop_words
for doc in cleaned[:20]:
    tokens = doc.split()
    assert not any(tok in stopwords for tok in tokens), f"Stopword found in: {tokens[:10]}"

# Vocabulary reduction study
vocab_changes = []
config_steps = [
    ('raw', {'lowercase':False,'remove_numbers':False,'remove_punctuation':False,
              'remove_stopwords':False,'lemmatize':False,'min_token_len':1}),
    ('lowercase', {'lowercase':True,'remove_numbers':False,'remove_punctuation':False,
                    'remove_stopwords':False,'lemmatize':False,'min_token_len':1}),
    ('+ no stopwords', {'lowercase':True,'remove_numbers':False,'remove_punctuation':False,
                         'remove_stopwords':True,'lemmatize':False,'min_token_len':1}),
    ('+ lemmatize', DEFAULT_CONFIG),
]
for name, cfg in config_steps:
    processed = preprocess_batch(sample[:100], cfg)
    if processed:
        vocab = set(' '.join(processed).split())
        vocab_changes.append({'step': name, 'vocab_size': len(vocab)})

vocab_df = pd.DataFrame(vocab_changes)
assert vocab_df['vocab_size'].is_monotonic_decreasing or len(vocab_df) == 0
print("✓ Exercise 1 passed")
print(vocab_df.to_string(index=False))

---
## Exercise 2 — Bag of Words & TF-IDF from Scratch

## The Math

**Term Frequency (TF):** $\text{tf}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total tokens in } d}$

**Inverse Document Frequency (IDF):** $\text{idf}(t) = \log\frac{N + 1}{\text{df}(t) + 1} + 1$ (sklearn's smooth IDF)

**TF-IDF:** $\text{tfidf}(t, d) = \text{tf}(t, d) \times \text{idf}(t)$, then L2-normalized per document.

**Why IDF matters:** Words that appear in every document ("the", "is") carry no discriminative information. IDF down-weights them. Words unique to a few documents get high weight.

**Task:** Implement TF-IDF from scratch. Verify against sklearn's TfidfVectorizer.

In [ ]:
class TFIDFScratch:
    """
    TF-IDF vectorizer from scratch.
    Matches sklearn's TfidfVectorizer(smooth_idf=True, sublinear_tf=False, norm='l2').
    """
    def __init__(self, max_features: int = None, min_df: int = 1):
        self.max_features = max_features
        self.min_df = min_df
        self.vocabulary_ = {}    # word → index
        self.idf_ = None          # idf values per term

    def _tokenize(self, text: str) -> list:
        """Simple whitespace tokenizer."""
        return text.lower().split()

    def fit(self, corpus: list) -> 'TFIDFScratch':
        """
        Build vocabulary and compute IDF from corpus.
        IDF = log((N+1)/(df+1)) + 1  (sklearn smooth IDF)
        If max_features: keep top max_features by document frequency.
        """
        # YOUR CODE HERE
        pass

    def transform(self, corpus: list) -> np.ndarray:
        """
        Compute TF-IDF matrix: shape (n_docs, vocab_size).
        TF = count(t in d) / total_tokens(d)
        TF-IDF = TF * IDF
        Then L2-normalize each row.
        """
        # YOUR CODE HERE
        pass

    def fit_transform(self, corpus: list) -> np.ndarray:
        return self.fit(corpus).transform(corpus)

# Use cleaned docs
small_corpus = cleaned[:200]
tfidf_scratch = TFIDFScratch(max_features=500)
X_scratch = tfidf_scratch.fit_transform(small_corpus)

In [ ]:
# --- ASSERTIONS vs sklearn ---
sk_tfidf = TfidfVectorizer(max_features=500, smooth_idf=True, sublinear_tf=False, norm='l2')
X_sk = sk_tfidf.fit_transform(small_corpus).toarray()

assert X_scratch is not None
assert X_scratch.shape == X_sk.shape, f"Shape mismatch: {X_scratch.shape} vs {X_sk.shape}"

# Row norms must be 0 or 1 (L2 normalized)
row_norms = np.linalg.norm(X_scratch, axis=1)
assert np.allclose(row_norms[row_norms > 0], 1.0, atol=1e-5), "Rows must be L2-normalized"

# IDF values should roughly match
scratch_idf = tfidf_scratch.idf_
assert (scratch_idf >= 1).all(), "Smooth IDF must be >= 1"

print(f"✓ Exercise 2 passed — TF-IDF shape: {X_scratch.shape}")
print(f"Vocab size: {len(tfidf_scratch.vocabulary_)}")

---
## Exercise 3 — N-grams and Subword Features

**Concept:** Unigrams lose word order. Bigrams and trigrams capture local context: "New York" is different from "new" + "york".

**Task:**
1. Train TF-IDF classifiers (LogisticRegression) on the newsgroups data with:
   - Unigrams only: `ngram_range=(1,1)`
   - Bigrams: `ngram_range=(1,2)`
   - Character n-grams: `analyzer='char_wb'`, `ngram_range=(3,5)` — captures morphology
2. Compare accuracy across all 3 on test set.
3. Extract the top 10 most informative features per class from the bigram model (highest TF-IDF coefficients in LogReg).
4. Return `ngram_comparison` DataFrame and `top_features_per_class` dict.

In [ ]:
# Preprocess all training and test docs
X_train_clean = preprocess_batch(X_train_raw, DEFAULT_CONFIG)
X_test_clean  = preprocess_batch(X_test_raw, DEFAULT_CONFIG)

# YOUR CODE HERE
ngram_comparison = None
top_features_per_class = None

In [ ]:
# --- ASSERTIONS ---
assert ngram_comparison is not None
assert len(ngram_comparison) == 3
acc_col = [c for c in ngram_comparison.columns if 'acc' in c.lower()][0]
assert (ngram_comparison[acc_col] > 0.6).all()

assert top_features_per_class is not None
assert len(top_features_per_class) == 4  # 4 classes
for cls, features in top_features_per_class.items():
    assert len(features) == 10, f"Need 10 features per class, got {len(features)}"

print("✓ Exercise 3 passed")
print(ngram_comparison.to_string(index=False))
print("\nTop features per class:")
for cls, features in list(top_features_per_class.items())[:2]:
    print(f"  {target_names[cls]}: {features}")

---
## Exercise 4 — Naive Bayes for Text: The Right Way

## The Math

**Multinomial Naive Bayes** for text:
$$P(c|d) \propto P(c) \prod_{t \in d} P(t|c)^{\text{count}(t,d)}$$

**Laplace smoothing:** $P(t|c) = \frac{\text{count}(t,c) + \alpha}{\sum_t \text{count}(t,c) + \alpha|V|}$

**Complement NB** (better for imbalanced classes): instead of modeling the target class, model the complement.

**Task:**
1. Train `MultinomialNB` and `ComplementNB` with various `alpha` values (0.01, 0.1, 1.0, 10.0).
2. Show how alpha affects accuracy — alpha is the Laplace smoothing parameter.
3. Implement `MultinomialNBScratch` from scratch to understand the log-probability computation.
4. Verify your implementation matches sklearn within 1% accuracy.

In [ ]:
class MultinomialNBScratch:
    """
    Multinomial Naive Bayes from scratch.
    Works with raw count matrices (not TF-IDF).
    """
    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha
        self.class_log_priors_ = None   # log P(c) per class
        self.feature_log_probs_ = None  # log P(t|c): shape (n_classes, n_features)
        self.classes_ = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'MultinomialNBScratch':
        """
        Estimate log-priors and log-likelihoods.
        X: count matrix (n_docs, n_features)
        log P(t|c) = log((count(t,c) + alpha) / (sum_t count(t,c) + alpha * |V|))
        """
        # YOUR CODE HERE
        pass

    def predict_log_proba(self, X: np.ndarray) -> np.ndarray:
        """
        Returns (n_docs, n_classes) unnormalized log-posteriors.
        log P(c|d) ∝ log P(c) + X @ log P(t|c).T
        """
        # YOUR CODE HERE
        pass

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]

# Build count matrix
count_vec = CountVectorizer(max_features=5000)
X_tr_counts = count_vec.fit_transform(X_train_clean).toarray()
X_te_counts = count_vec.transform(X_test_clean).toarray()

# YOUR CODE HERE: compare alpha values, verify scratch implementation
alpha_comparison = None

In [ ]:
# --- ASSERTIONS ---
nb_scratch = MultinomialNBScratch(alpha=1.0)
nb_scratch.fit(X_tr_counts, y_train)
preds_scratch = nb_scratch.predict(X_te_counts)

nb_sk = MultinomialNB(alpha=1.0)
nb_sk.fit(X_tr_counts, y_train)
preds_sk = nb_sk.predict(X_te_counts)

from sklearn.metrics import accuracy_score
acc_scratch = accuracy_score(y_test, preds_scratch)
acc_sk = accuracy_score(y_test, preds_sk)
assert abs(acc_scratch - acc_sk) < 0.01, f"Accuracy gap: {acc_scratch:.4f} vs {acc_sk:.4f}"

if alpha_comparison is not None:
    assert len(alpha_comparison) == 4

print(f"✓ Exercise 4 passed — Scratch NB: {acc_scratch:.4f} | sklearn: {acc_sk:.4f}")

---
## Exercise 5 — Text Classification Benchmark

**Task:** Systematic comparison of classical NLP classifiers.

1. Build pipelines for: MultinomialNB, ComplementNB, LogisticRegression, LinearSVC.
2. For each: TF-IDF preprocessing (unigrams + bigrams, max_features=10000).
3. Evaluate: accuracy, macro F1, per-class F1, training time.
4. Find the most confused class pairs using the confusion matrix.
5. Return `classifier_benchmark` DataFrame.

In [ ]:
import time

# YOUR CODE HERE
classifier_benchmark = None

In [ ]:
# --- ASSERTIONS ---
assert classifier_benchmark is not None
assert len(classifier_benchmark) == 4
acc_col = [c for c in classifier_benchmark.columns if 'acc' in c.lower()][0]
assert (classifier_benchmark[acc_col] > 0.70).all(), "All classifiers should exceed 70% accuracy"
print("✓ Exercise 5 passed")
print(classifier_benchmark.to_string(index=False))

---
## Exercise 6 — Linguistic Features with spaCy

**Concept:** TF-IDF loses linguistic structure. spaCy's POS tags, dependency parse, and NER can create richer features.

**Task:** For each document, extract:
1. `pos_dist`: distribution of POS tags (noun%, verb%, adj%, adv%) — normalized
2. `avg_sentence_length`: mean tokens per sentence
3. `entity_density`: named entities per 100 tokens
4. `entity_types`: one-hot of whether document contains PERSON, ORG, GPE, DATE entities
5. `avg_token_depth`: mean dependency tree depth of tokens

Combine these 10+ features with TF-IDF and test if they improve classification.

In [ ]:
def extract_linguistic_features(text: str) -> dict:
    """
    Extract POS, NER, syntactic features from a document.
    Returns flat dict of numeric features.
    """
    # YOUR CODE HERE
    pass

# Apply to first 300 docs (spaCy parsing is slow)
sample_docs = X_train_raw[:300]
ling_features = [extract_linguistic_features(doc) for doc in sample_docs]
ling_df = pd.DataFrame(ling_features)
print(f"Linguistic features: {list(ling_df.columns)}")
ling_df.head()

In [ ]:
# --- ASSERTIONS ---
assert len(ling_features) == 300
assert all(isinstance(f, dict) for f in ling_features)
# Required features
required = ['noun_pct', 'verb_pct', 'avg_sentence_length', 'entity_density']
for feat in required:
    assert feat in ling_df.columns, f"Missing feature: {feat}"
# POS percentages should sum to ~1
pos_cols = [c for c in ling_df.columns if c.endswith('_pct')]
if len(pos_cols) >= 2:
    row_sums = ling_df[pos_cols].sum(axis=1)
    assert (row_sums > 0).sum() > 200, "Most docs should have POS features"
print(f"✓ Exercise 6 passed — {len(ling_df.columns)} linguistic features extracted")

---
## Exercise 7 — Keyword Extraction: TF-IDF vs TextRank

**Task:** Implement two keyword extraction methods and compare.

**Method 1: TF-IDF keyword extraction**
Fit TF-IDF on the whole corpus. For a new document, return the tokens with the highest TF-IDF scores.

**Method 2: TextRank (simplified)**
Build a word co-occurrence graph: words within a sliding window of size $k$ are connected. Run PageRank on this graph. Top-ranked nodes are keywords.

1. Implement both methods.
2. For 5 sample documents, compare top-5 keywords from each method.
3. Compute overlap between the two methods' top-5 keywords.

In [ ]:
def tfidf_keywords(doc: str, fitted_vectorizer: TfidfVectorizer,
                    top_k: int = 5) -> list:
    """
    Extract top-k keywords by TF-IDF score.
    """
    # YOUR CODE HERE
    pass

def textrank_keywords(text: str, window: int = 3,
                       top_k: int = 5, n_iter: int = 30,
                       d: float = 0.85) -> list:
    """
    Simplified TextRank keyword extraction.
    1. Tokenize and clean (lowercase, remove stopwords/punct)
    2. Build co-occurrence graph: edges between tokens within window
    3. PageRank: score[i] = (1-d) + d * sum_j(score[j] / out_degree[j])
    4. Return top-k tokens by PageRank score
    """
    # YOUR CODE HERE
    pass

# Fit TF-IDF on training corpus
kw_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,1))
kw_vectorizer.fit(X_train_clean)

# Compare on 5 documents
comparison_docs = X_train_raw[:5]
keyword_comparison = []
for i, doc in enumerate(comparison_docs):
    clean_doc = preprocess_text(doc, DEFAULT_CONFIG)
    tfidf_kw = tfidf_keywords(clean_doc, kw_vectorizer)
    textrank_kw = textrank_keywords(doc)
    if tfidf_kw and textrank_kw:
        overlap = len(set(tfidf_kw) & set(textrank_kw))
        keyword_comparison.append({
            'doc_idx': i,
            'tfidf': tfidf_kw,
            'textrank': textrank_kw,
            'overlap': overlap
        })

kw_df = pd.DataFrame(keyword_comparison)
kw_df

In [ ]:
# --- ASSERTIONS ---
assert len(keyword_comparison) > 0
for row in keyword_comparison:
    assert len(row['tfidf']) <= 5
    assert len(row['textrank']) <= 5
    assert 0 <= row['overlap'] <= 5
print(f"✓ Exercise 7 passed")
print(f"Mean keyword overlap: {kw_df['overlap'].mean():.2f}/5")

---
## Exercise 8 — Topic Modeling: LDA

## The Math

**LDA (Latent Dirichlet Allocation)** assumes each document is a mixture of topics, and each topic is a distribution over words:
$$p(\text{word}|\text{doc}) = \sum_k p(\text{word}|\text{topic}_k) \cdot p(\text{topic}_k|\text{doc})$$

Documents close in topic space are semantically similar — even if they share no words.

**Task:**
1. Fit `LatentDirichletAllocation` on the training corpus (4 categories → try 4 and 8 topics).
2. For each topic, extract the top 10 words.
3. Map discovered topics to the ground-truth categories: which topic aligns with which newsgroup?
4. Use topic distributions as features for a classifier. Does it beat TF-IDF?
5. Return `lda_result`: dict with topics, topic_words, classifier_accuracy.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# YOUR CODE HERE
lda_result = None

In [ ]:
# --- ASSERTIONS ---
assert lda_result is not None
assert 'topic_words' in lda_result
assert len(lda_result['topic_words']) in [4, 8]
for topic_words in lda_result['topic_words']:
    assert len(topic_words) == 10
print("✓ Exercise 8 passed")
print("\nTop words per topic:")
for i, words in enumerate(lda_result['topic_words']):
    print(f"  Topic {i}: {', '.join(words)}")

---
## Exercise 9 — Text Similarity & Document Retrieval

**Task:** Build a simple document retrieval system using TF-IDF cosine similarity.

1. Build a TF-IDF index over the training corpus.
2. Implement `search(query, top_k=5)` that returns the most similar documents.
3. Implement `find_similar_documents(doc_idx, top_k=5)` — given a document, find the most similar others.
4. Verify: similar documents should be in the same newsgroup category (precision@5).
5. Implement **BM25** (an improved TF-IDF for retrieval) and compare precision@5.

In [ ]:
class TextSearchIndex:
    """
    TF-IDF + cosine similarity document search.
    """
    def __init__(self, max_features: int = 10000):
        self.vectorizer = TfidfVectorizer(max_features=max_features)
        self.doc_matrix = None
        self.documents = None

    def index(self, documents: list) -> 'TextSearchIndex':
        """Build TF-IDF index."""
        # YOUR CODE HERE
        pass

    def search(self, query: str, top_k: int = 5) -> list:
        """
        Return (doc_idx, score) pairs for top_k most similar docs.
        """
        # YOUR CODE HERE
        pass

    def find_similar(self, doc_idx: int, top_k: int = 5) -> list:
        """
        Return top_k most similar documents to doc at doc_idx.
        Exclude the query document itself.
        """
        # YOUR CODE HERE
        pass

index = TextSearchIndex(max_features=10000)
index.index(X_train_clean)

In [ ]:
# --- ASSERTIONS ---
results = index.search('space shuttle nasa orbit', top_k=5)
assert results is not None and len(results) == 5
assert all(isinstance(r[0], (int, np.integer)) for r in results)
assert all(0 <= r[1] <= 1 for r in results), "Cosine similarity must be in [0,1]"

# Similar docs should mostly be in same category
similar = index.find_similar(0, top_k=5)
assert similar is not None and len(similar) == 5
# Category of query doc
query_cat = y_train[0]
similar_cats = [y_train[r[0]] for r in similar]
precision_at_5 = sum(c == query_cat for c in similar_cats) / 5
assert precision_at_5 >= 0.4, f"Precision@5 too low: {precision_at_5:.2f}"

print(f"✓ Exercise 9 passed — Precision@5: {precision_at_5:.2f}")

---
## Exercise 10 — Capstone: NLP Classification Pipeline

**Spec:** Build the best possible classical NLP classifier on the full 20 Newsgroups dataset (all 20 categories).

1. Load all 20 categories.
2. Build a rich feature set combining:
   - TF-IDF (word + char n-grams)
   - At least 3 additional hand-crafted features
3. Train and tune a LinearSVC classifier.
4. Report: per-class F1, macro F1, most confused pairs.
5. Perform error analysis: show 3 misclassified examples and explain why they're hard.
6. Return `capstone_result` dict.

In [ ]:
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer

# Load full 20 newsgroups
news20_train = fetch_20newsgroups(subset='train', remove=('headers','footers','quotes'))
news20_test  = fetch_20newsgroups(subset='test',  remove=('headers','footers','quotes'))

# YOUR CODE HERE
capstone_result = None

In [ ]:
# --- ASSERTIONS ---
assert capstone_result is not None
required = ['macro_f1', 'per_class_f1', 'confused_pairs', 'error_examples']
for k in required:
    assert k in capstone_result, f"Missing: {k}"
assert capstone_result['macro_f1'] > 0.60, f"Macro F1 too low: {capstone_result['macro_f1']:.4f}"
assert len(capstone_result['per_class_f1']) == 20
assert len(capstone_result['confused_pairs']) >= 3
assert len(capstone_result['error_examples']) >= 3
print(f"✓ Exercise 10 passed — 20 Newsgroups Macro F1: {capstone_result['macro_f1']:.4f}")